# Sentence-BERT + SVM

Sentence-BERT (SBERT) is a pre-trained transformer, fine-tuned BERT on sentence-similarity tasks, that converts any sentence or paragraph into a fixed-length dense vector (typically 384 or 768 dimensions). Unlike TF-IDF, which produces sparse vectors with one dimension per word, SBERT produces dense vectors where each dimension is a learned semantic feature.

**Two sentences with similar meaning get similar vectors even if they share no words.**

This approach is a "baseline" because it uses a pre-trained transformer but does not adapt it to this specific task.

Model options:
- all-mpnet-base-v2 — the default strong choice as of 2026. 768-dim vectors.
- all-MiniLM-L6-v2 — smaller and faster. 384-dim. About 1% lower quality but 5x faster.
- BAAI/bge-large-en-v1.5 — top of the embedding leaderboards. 1024-dim. Slower to encode.

**Long documents**. SBERT models have a maximum input length (typically 384 or 512 tokens). Some TherapistQA patient questions are longer than this. The model will truncate them silently.

TherapistQA's label distribution is heavily skewed - using weighted F1 because it weights each class's F1 by its support (number of examples).

**Prepare *TherapistQA* dataset**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load preprocessed TherapistQA from Stage 0
df = pd.read_csv('../../processed/therapistqa.csv')

# Split - use the same split every time for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    df['patient_question'],
    df['dominant_distortion'],
    test_size=0.2,
    stratify=df['dominant_distortion'],
    random_state=42
)

**Prepare *PatternReframe* dataset**

In [1]:
import pandas as pd

# Load PatternReframe splits from Stage 0 preprocessing
df_train = pd.read_csv('../../processed/patternreframe_train.csv')
df_test = pd.read_csv('../../processed/patternreframe_test.csv')

X_train = df_train['thought']
y_train = df_train['pattern']
X_test = df_test['thought']
y_test = df_test['pattern']

print(f"PatternReframe train: {len(X_train)} records")
print(f"PatternReframe test: {len(X_test)} records")
print(f"Train label distribution:")
print(y_train.value_counts())
print(f"\nTest label distribution:")
print(y_test.value_counts())

PatternReframe train: 1920 records
PatternReframe test: 6807 records
Train label distribution:
pattern
Catastrophizing                                                    200
Jumping to conclusions: Fortune-telling                            199
Overgeneralization                                                 197
Jumping to conclusions: mind reading                               197
Labeling and mislabeling                                           196
Discounting the positive                                           194
Black-and-white or polarized thinking / All or nothing thinking    188
Personalization                                                    185
Mental filtering                                                   185
Should statements                                                  179
Name: count, dtype: int64

Test label distribution:
pattern
Catastrophizing                                                    726
Jumping to conclusions: Fortune-telling                 

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# Load pre-trained SBERT model
# all-mpnet-base-v2 (the strongest general-purpose SBERT model as of 2026, produces 768-dimensional embeddings)
# all-MiniLM-L6-v2 (smaller and faster, 384-dim. About 1% lower quality but 5x faster.)
# BAAI/bge-large-en-v1.5 (top of the embedding leaderboards. 1024-dim. Slower to encode.)
model = SentenceTransformer('BAAI/bge-large-en-v1.5')

# Encode all documents into vectors
# This takes a few minutes on CPU, seconds on GPU
X_train_emb = model.encode(X_train.tolist(), show_progress_bar=True)
X_test_emb = model.encode(X_test.tolist(), show_progress_bar=True)

# Train SVM on the embeddings
svm = LinearSVC(C=1.0, class_weight='balanced', max_iter=5000)
svm.fit(X_train_emb, y_train)

# Evaluate
y_pred = svm.predict(X_test_emb)
print(classification_report(y_test, y_pred, digits=3))

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 213/213 [00:47<00:00,  4.50it/s]


                                                                 precision    recall  f1-score   support

Black-and-white or polarized thinking / All or nothing thinking      0.370     0.317     0.342       665
                                                Catastrophizing      0.540     0.556     0.548       726
                                       Discounting the positive      0.676     0.667     0.671       681
                        Jumping to conclusions: Fortune-telling      0.381     0.368     0.375       711
                           Jumping to conclusions: mind reading      0.538     0.717     0.614       699
                                       Labeling and mislabeling      0.535     0.502     0.518       669
                                               Mental filtering      0.515     0.476     0.494       660
                                             Overgeneralization      0.379     0.333     0.354       688
                                                Person

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
